<a href="https://colab.research.google.com/github/yilinw762/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026_09_25_%E2%80%94_Cleaning_Gauntlet_%E2%80%94_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Gauntlet

**Lab — 2026-09-25 · Fall 2026**  

---

## Lab 05 — Cleaning Gauntlet

Three hundred rows, generated messy. This is the first dataset in the course you cannot eyeball, which means you have to work from counts and assertions rather than from looking at the table and deciding it seems fine.

Deliverables: a clean frame, a decision log, a set of assertions that pass, and one business number at the end — revenue by category — that you would be willing to defend.

Keep the log as you go. Reconstructing it afterward is much harder than writing one line per step, and the write-up at the end depends on it.

### The log

Run this first, then call `log(...)` after each cleaning step.

In [1]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

In [2]:
import pandas as pd, numpy as np
from io import StringIO
rng = np.random.default_rng(5)
items = ['Cheeseburger','cheese burger','Foam Finger','foam finger','Rain Poncho','rain poncho']
cats = ['Food','food','Merch','Apparel','RainGear','rain-gear']
rows = []
for i in range(300):
    rows.append({
        'order_id': i,
        'item': rng.choice(items),
        'category': rng.choice(cats),
        'qty': rng.choice([1,2,3,-1,np.nan], p=[.5,.25,.15,.05,.05]),
        'price': rng.choice(['$7.50','7.5','$12.00','24','6.0']),
    })
df = pd.DataFrame(rows)
df = pd.concat([df, df.sample(15, random_state=1)])  # inject dupes
df.head()

,order_id,item,category,qty,price
0,0,Rain Poncho,RainGear,3.0,$12.00
1,1,foam finger,Apparel,1.0,7.5
2,2,cheese burger,Merch,1.0,$7.50
3,3,Cheeseburger,Food,NaN,$7.50
4,4,cheese burger,Apparel,1.0,7.5


### TODO 1 — drop duplicates

In [3]:
# TODO
# Use parsed values only to measure revenue consistently at every stage.
# Missing quantities are excluded from these diagnostic sums, not imputed.
def revenue_total(frame):
    prices = pd.to_numeric(frame['price'].astype(str).str.replace('$', '', regex=False), errors='raise')
    quantities = pd.to_numeric(frame['qty'], errors='coerce')
    return float((prices * quantities).sum())

REVENUE_AUDIT = []
def audit(step, before):
    after = revenue_total(df)
    REVENUE_AUDIT.append({'step': step, 'before': before, 'after': after, 'change': after - before})

raw_df = df.copy()
before = revenue_total(df)
rows_before = len(df)
df = df.drop_duplicates().copy()
log('1 — duplicates', 'Remove exact duplicate rows; retain one copy of each order.', rows_before - len(df))
audit('1 — duplicates', before)

[1 — duplicates] Remove exact duplicate rows; retain one copy of each order. (15 row(s))


### TODO 2 — clean `price` -> float

In [4]:
# TODO
before = revenue_total(df)
df['price'] = pd.to_numeric(df['price'].astype(str).str.replace('$', '', regex=False).str.strip(), errors='raise').astype(float)
log('2 — price', 'Remove dollar signs and convert all prices to float; do not change their values.', len(df))
audit('2 — price', before)

[2 — price] Remove dollar signs and convert all prices to float; do not change their values. (300 row(s))


### TODO 3 — `qty` -> numeric, drop rows with missing/negative qty

In [5]:
# TODO
before = revenue_total(df)
df['qty'] = pd.to_numeric(df['qty'], errors='coerce')
missing_qty = df['qty'].isna()
negative_qty = df['qty'].lt(0)
removed_qty_rows = df.loc[missing_qty | negative_qty].copy()
print('Missing qty:', int(missing_qty.sum()))
print('Negative qty:', int(negative_qty.sum()))
df = df.loc[~(missing_qty | negative_qty)].copy()
log('3 — qty', 'Drop missing quantities rather than guess; drop negative quantities as required by the lab.', int(missing_qty.sum() + negative_qty.sum()))
audit('3 — qty', before)

Missing qty: 12
Negative qty: 13
[3 — qty] Drop missing quantities rather than guess; drop negative quantities as required by the lab. (25 row(s))


### TODO 4 — canonicalize `item`

Six spellings, three real products. Start by listing what you actually have, then build the mapping from that list rather than from memory.

```python
print(df['item'].value_counts())
ITEM_MAP = {...}
```

In [6]:
# TODO: inspect the variants, build a mapping dict, apply it, log the collapse
print(df['item'].value_counts())
ITEM_MAP = {
    'Cheeseburger': 'Cheeseburger',
    'cheese burger': 'Cheeseburger',
    'Foam Finger': 'Foam Finger',
    'foam finger': 'Foam Finger',
    'Rain Poncho': 'Rain Poncho',
    'rain poncho': 'Rain Poncho'
}
assert set(df['item']).issubset(ITEM_MAP)
before = revenue_total(df)
clean_items = df['item'].map(ITEM_MAP)
changed = int(df['item'].ne(clean_items).sum())
df['item'] = clean_items
log('4 — item', 'Collapse six observed spellings into three canonical product names.', changed)
audit('4 — item', before)
print(df['item'].value_counts())

item
Foam Finger      57
Rain Poncho      49
cheese burger    44
Cheeseburger     43
rain poncho      42
foam finger      40
Name: count, dtype: int64
[4 — item] Collapse six observed spellings into three canonical product names. (126 row(s))
item
Foam Finger     97
Rain Poncho     91
Cheeseburger    87
Name: count, dtype: int64


### TODO 5 — normalize `category`

Same approach. Note that `Apparel` and `Merch` are a business decision, not a string problem — decide and log it.

In [7]:
# TODO
print(df['category'].value_counts())
CATEGORY_MAP = {
    'Food': 'Food', 'food': 'Food',
    'Merch': 'Merch', 'Apparel': 'Merch',
    'RainGear': 'RainGear', 'rain-gear': 'RainGear'
}
assert set(df['category']).issubset(CATEGORY_MAP)
before = revenue_total(df)
apparel_revenue = revenue_total(df.loc[df['category'].eq('Apparel')])
clean_categories = df['category'].map(CATEGORY_MAP)
changed = int(df['category'].ne(clean_categories).sum())
df['category'] = clean_categories
log('5 — category', 'Normalize Food and RainGear spellings; treat Apparel as part of the broader Merch category. Preserve recorded category assignments.', changed)
audit('5 — category', before)
print(df['category'].value_counts())

category
Food         51
Merch        51
rain-gear    45
food         44
Apparel      43
RainGear     41
Name: count, dtype: int64
[5 — category] Normalize Food and RainGear spellings; treat Apparel as part of the broader Merch category. Preserve recorded category assignments. (132 row(s))
category
Food        95
Merch       94
RainGear    86
Name: count, dtype: int64


### TODO 6 — prove it's clean

**TODO:** uncomment these and add two more assertions of your own — one about the item names and one about the categories.

In [8]:
assert df.duplicated().sum() == 0
assert df['qty'].min() >= 1
assert df['price'].dtype == float
assert df['item'].isin({'Cheeseburger', 'Foam Finger', 'Rain Poncho'}).all()
assert df['category'].isin({'Food', 'Merch', 'RainGear'}).all()
assert df[['order_id', 'item', 'category', 'qty', 'price']].notna().all().all()
assert df['order_id'].is_unique
assert np.isfinite(df[['qty', 'price']].to_numpy()).all()
assert df['price'].gt(0).all()
print('All assertions passed.')
print('clean:', df.shape)

All assertions passed.
clean: (275, 5)


### TODO 7 — the number you would report

**TODO:** add a `revenue` column, then print revenue by category, highest first, plus the overall total. Round money to two decimals.

Then, in one sentence, state what you would tell a vendor to stock more of.

In [9]:
# TODO
df['revenue'] = df['qty'] * df['price']
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False)
print('Revenue by category:')
print(revenue_by_category.to_string(float_format=lambda amount: f'${amount:,.2f}'))
print(f'Overall total: ${df["revenue"].sum():,.2f}')
assert np.isclose(revenue_by_category.sum(), df['revenue'].sum())
print('\nRevenue audit:')
print(pd.DataFrame(REVENUE_AUDIT).to_string(index=False, float_format=lambda amount: f'{amount:,.2f}'))

Revenue by category:
category
Food       $1,656.00
Merch      $1,572.00
RainGear   $1,512.00
Overall total: $4,740.00

Revenue audit:
          step   before    after  change
1 — duplicates 4,852.50 4,594.50 -258.00
     2 — price 4,594.50 4,594.50    0.00
       3 — qty 4,594.50 4,740.00  145.50
      4 — item 4,740.00 4,740.00    0.00
  5 — category 4,740.00 4,740.00    0.00


**What I would tell the vendor:** stock more Food because it has the highest recorded category revenue ($1,656.00), but verify item-to-category assignments and unit demand before placing an order.

### TODO 8 — read back your log

In [10]:
import pandas as pd
pd.DataFrame(DECISIONS)

,step,decision,rows
0,1 — duplicates,Remove exact duplicate rows; retain one copy o...,15
1,2 — price,Remove dollar signs and convert all prices to ...,300
2,3 — qty,Drop missing quantities rather than guess; dro...,25
3,4 — item,Collapse six observed spellings into three can...,126
4,5 — category,Normalize Food and RainGear spellings; treat A...,132


### Write-up

Two parts.

**a)** Which cleaning step changed your revenue total the most? Give the number before and after that step, not a description.

**b)** Pick one decision you made where a reasonable person could have chosen differently. State the other choice, what it would have done to your reported revenue, and why you went the way you did.

### a) Which cleaning step changed revenue the most?

Dropping duplicates decreased revenue from **\$4,852.50 to \$4,594.50**, a decrease of **\$258.00**. This was the largest change. Removing missing or negative quantities then increased the total to **\$4,740.00**. These comparisons exclude missing quantities from the sums.

### b) What decision could reasonably have been made differently?

I combined **Apparel** with **Merch** because apparel is a type of merchandise. Another reasonable choice would be to keep them separate. Apparel would then have **\$715.50** in revenue, and Merch would decrease from **\$1,572.00 to \$856.50**. The overall total would remain **\$4,740.00**. I combined them to simplify the report, but separating them would help the vendor track clothing sales.